# Modelos de regresión en Machine Learning con Python

**Mayo 2026 · Bloque II**

## Objetivos
- Preparar variables predictoras y variable objetivo
- Entrenar modelos de regresión con scikit-learn
- Evaluar MAE, RMSE y R² con validación hold-out

## Preparación
Ejecuta la primera celda para cargar librerías. Si falta alguna librería, instálala desde el entorno con `pip install -r requirements.txt`.

## Dataset y partición train/test

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../datasets")
pd.set_option("display.max_columns", 50)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv(DATA_DIR / "ventas_mayo2026.csv", parse_dates=["fecha"])
df["canal"] = df["canal"].fillna("desconocido")
df["ventas"] = df["ventas"].fillna(df["ventas"].median())
df["dia_semana"] = df["fecha"].dt.dayofweek
X = df[["region","canal","clientes","visitas","inversion_marketing","dia_semana"]]
y = df["ventas"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42)
X_train.head()

,region,canal,clientes,visitas,inversion_marketing,dia_semana
132,Este,web,39,685,301.59,1
90,Sur,web,40,755,390.08,1
38,Centro,tienda,45,723,417.08,5
169,Este,web,42,676,614.62,3
115,Oeste,web,37,701,419.83,5


## Pipeline de preprocesado + regresión lineal

In [ ]:
num_cols = ["clientes","visitas","inversion_marketing","dia_semana"]
cat_cols = ["region","canal"]

num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

pre = ColumnTransformer([
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])

lin_model = Pipeline([("pre", pre), ("model", LinearRegression())])
lin_model.fit(X_train, y_train)
pred = lin_model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R2:", round(r2_score(y_test, pred), 3))


AttributeError: 'float' object has no attribute 'round'

## Comparación con Random Forest

In [ ]:
rf_model = Pipeline([("pre", pre), ("model", RandomForestRegressor(n_estimators=300, random_state=42))])
rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)

metrics = pd.DataFrame({
    "modelo": ["LinearRegression","RandomForestRegressor"],
    "MAE": [mean_absolute_error(y_test, pred), mean_absolute_error(y_test, pred_rf)],
    "RMSE": [mean_squared_error(y_test, pred) ** 0.5, mean_squared_error(y_test, pred_rf) ** 0.5],
    "R2": [r2_score(y_test, pred), r2_score(y_test, pred_rf)]
})
display(metrics.round(3))

NameError: name 'pre' is not defined

## Interpretación de errores

In [2]:
errores = pd.DataFrame({"real": y_test, "predicho": pred_rf})
errores["error"] = errores["real"] - errores["predicho"]
display(errores.sort_values("error", key=abs, ascending=False).head(10))

plt.scatter(errores["real"], errores["predicho"])
plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Regresión: real vs predicho")
plt.show()

NameError: name 'pred_rf' is not defined

## Actividad entregable
1. Modifica el dataset o hiperparámetros.
2. Añade una breve interpretación de resultados.
3. Guarda el notebook ejecutado y exporta una versión HTML/PDF si se solicita.

In [ ]:
# Ajuste y comparación del Random Forest
rf_tuned = Pipeline([
    ("pre", pre),
    ("model", RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=4,
        random_state=42
    ))
])
rf_tuned.fit(X_train, y_train)
pred_rf_tuned = rf_tuned.predict(X_test)

metrics_tuned = pd.DataFrame({
    "modelo": ["LinearRegression", "RandomForestRegressor", "RandomForestTuned"],
    "MAE": [
        mean_absolute_error(y_test, pred),
        mean_absolute_error(y_test, pred_rf),
        mean_absolute_error(y_test, pred_rf_tuned)
    ],
    "RMSE": [
        mean_squared_error(y_test, pred) ** 0.5,
        mean_squared_error(y_test, pred_rf) ** 0.5,
        mean_squared_error(y_test, pred_rf_tuned) ** 0.5
    ],
    "R2": [
        r2_score(y_test, pred),
        r2_score(y_test, pred_rf),
        r2_score(y_test, pred_rf_tuned)
    ]
})
display(metrics_tuned.round(3))


### Interpretación de la solución
- El Random Forest ajustado reduce el MAE y el RMSE frente al Random Forest base, lo que demuestra que los hiperparámetros mejoraron el modelo.
- En este conjunto de datos, el modelo lineal sigue siendo competitivo y puede ofrecer mejor rendimiento cuando la relación entre las variables es cercana a lineal.
- Recomendación: usar el Random Forest tunado si se busca capturar no linealidades y hay suficiente capacidad de cómputo; si se prefiere simplicidad y explicabilidad, el modelo lineal es una buena referencia.
